## Cell 1: Data Loading & Sequence Generation

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Hardware Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on: {device}")

# 2. Load Parquet Data
print("Loading Parquet data...")
df = pd.read_parquet("deep_learning_data.parquet")

# Function to unpack PySpark Vectors into pure Python numbers
def extract_numbers(row):
    # If PySpark saved it as a dictionary, pull out the 'values' array
    if isinstance(row, dict) and 'values' in row:
        return row['values']
    # Otherwise, just return it as a normal list
    return list(row)

print("Unpacking PySpark objects into pure math numbers...")
clean_features = df['features'].apply(extract_numbers).tolist()

# Convert to strict float32 numbers so PyTorch is happy
X_raw = np.array(clean_features, dtype=np.float32) 
y_raw = df['score'].values.astype(np.float32)

# 3. Sliding Window Function
def create_sequences(X, y, window_size=7):
    """Converts continuous time-series into windowed sequences."""
    Xs, ys = [], []
    for i in range(len(X) - window_size):
        Xs.append(X[i:(i + window_size)]) # Input: Past 7 days
        ys.append(y[i + window_size])     # Target: Day 8
    return np.array(Xs), np.array(ys)

print("Processing time-series sequences...")
X_seq, y_seq = create_sequences(X_raw, y_raw, window_size=7)

# 4. Chronological Train/Test Split (80/20)
# 70% train, 15% val, 15% test
train_end = int(len(X_seq) * 0.70)
val_end   = int(len(X_seq) * 0.85)

X_train = X_seq[:train_end]
y_train = y_seq[:train_end]

X_val   = X_seq[train_end:val_end]
y_val   = y_seq[train_end:val_end]

X_test  = X_seq[val_end:]
y_test  = y_seq[val_end:]

# 5. PyTorch Dataset & DataLoader Initialization
class TimeSeriesDataset(Dataset):
    def __init__(self, features, targets):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets, dtype=torch.float32).reshape(-1, 1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# Batch size is set to 256 to optimize GPU memory utilization for 2.7M rows
train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=256, shuffle=True)
test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=256, shuffle=False)
val_loader = DataLoader(TimeSeriesDataset(X_val, y_val), batch_size=256, shuffle=False)

Executing on: cuda
Loading Parquet data...
Unpacking PySpark objects into pure math numbers...
Processing time-series sequences...


In [2]:
# Print the exact number of sequences (rows) in each set
print(f"Train:      {len(X_train):,}")
print(f"Validation: {len(X_val):,}")
print(f"Test:       {len(X_test):,}")

Train:      1,929,752
Validation: 413,518
Test:       413,519


## Cell 2: The GRU Brain

In [3]:
import torch.nn as nn

class DroughtPredictorGRU(nn.Module):
    def __init__(self, input_dim=18, hidden_dim=64, output_dim=1, dropout=0.3):
        super(DroughtPredictorGRU, self).__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, 
                          batch_first=True, dropout=dropout)  # ← dropout inside GRU
        self.dropout = nn.Dropout(dropout)                    # ← dropout before linear
        self.linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_step = gru_out[:, -1, :]
        last_step = self.dropout(last_step)   # ← apply dropout here
        return self.linear(last_step)

# Initialize model, loss function, and optimizer
model = DroughtPredictorGRU().to(device)
criterion = nn.MSELoss() # Grader (Mean Squared Error)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Optimizer

print("GRU Brain created successfully!")

c:\Users\Panhavath\anaconda3\envs\FOR_ML\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


GRU Brain created successfully!


## Cell 3: The Training Loop

In [4]:
epochs = 50          # high ceiling — early stopping will cut it short
patience = 5         # stop if no improvement for 5 epochs in a row
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

print("Initiating Model Training...")
for epoch in range(epochs):

    # --- PHASE 1: Training ---
    model.train()
    total_train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    # --- PHASE 2: Validation ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- Early Stopping Check ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()  # save best weights
        print(f"  ✓ New best model saved!")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Load best weights back into model
model.load_state_dict(best_model_state)
print(f"Training done! Best val loss: {best_val_loss:.4f}")

Initiating Model Training...
Epoch 01/50 | Train Loss: 0.9017 | Val Loss: 0.9619
  ✓ New best model saved!
Epoch 02/50 | Train Loss: 0.7513 | Val Loss: 0.9028
  ✓ New best model saved!
Epoch 03/50 | Train Loss: 0.6573 | Val Loss: 0.8807
  ✓ New best model saved!
Epoch 04/50 | Train Loss: 0.6067 | Val Loss: 0.8543
  ✓ New best model saved!
Epoch 05/50 | Train Loss: 0.5757 | Val Loss: 0.8428
  ✓ New best model saved!
Epoch 06/50 | Train Loss: 0.5539 | Val Loss: 0.8590
  No improvement (1/5)
Epoch 07/50 | Train Loss: 0.5383 | Val Loss: 0.8406
  ✓ New best model saved!
Epoch 08/50 | Train Loss: 0.5260 | Val Loss: 0.8424
  No improvement (1/5)
Epoch 09/50 | Train Loss: 0.5163 | Val Loss: 0.8843
  No improvement (2/5)
Epoch 10/50 | Train Loss: 0.5080 | Val Loss: 0.8591
  No improvement (3/5)
Epoch 11/50 | Train Loss: 0.5019 | Val Loss: 0.8443
  No improvement (4/5)
Epoch 12/50 | Train Loss: 0.4962 | Val Loss: 0.8376
  ✓ New best model saved!
Epoch 13/50 | Train Loss: 0.4913 | Val Loss: 0.852

In [13]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

model = model.to(device)
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_batch.numpy())

all_preds = np.array(all_preds).flatten()
all_targets = np.array(all_targets).flatten()

mse  = mean_squared_error(all_targets, all_preds)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(all_targets, all_preds)
r2   = r2_score(all_targets, all_preds)

print(f"Val MSE:  {mse:.4f}")
print(f"Val RMSE: {rmse:.4f}")
print(f"Val MAE:  {mae:.4f}")
print(f"Val R²:   {r2:.4f}")

Val MSE:  0.8450
Val RMSE: 0.9192
Val MAE:  0.6295
Val R²:   0.5019


## How to Save your Model

In [6]:
import torch

# Define the name of your save file (ending in .pth is standard for PyTorch)
save_path = "drought_gru_model.pth"

# Save the model's learned memory
torch.save(model.state_dict(), save_path)

print(f"Model successfully saved to: {save_path}")

Model successfully saved to: drought_gru_model.pth


## Bonus: How to Load the Model Later

In [7]:
# 1. Build an empty brain
loaded_model = DroughtPredictorGRU().to(device)

# 2. Load your saved memory into the empty brain
loaded_model.load_state_dict(torch.load("drought_gru_model.pth"))

# 3. Put it in testing mode (so it doesn't try to learn anymore)
loaded_model.eval()

print("Model successfully loaded and ready to make predictions!")

Model successfully loaded and ready to make predictions!


c:\Users\Panhavath\anaconda3\envs\FOR_ML\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


In [8]:
import torch
import torch.nn as nn
import numpy as np

# ---------------------------------------------------------
# STEP 1: Provide the Blueprint
# The app needs to know what the brain looks like before loading it.
# This must perfectly match the code you used to train it!
# ---------------------------------------------------------
class DroughtPredictorGRU(nn.Module):
    def __init__(self, input_dim=18, hidden_size=64, output_dim=1):
        super(DroughtPredictorGRU, self).__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_day_output = gru_out[:, -1, :]
        return self.linear(last_day_output)

# ---------------------------------------------------------
# STEP 2: Wake up the AI and Load the Memory
# ---------------------------------------------------------
# 1. Create an empty brain
model = DroughtPredictorGRU()

# 2. Load your saved .pth file into the brain
# (Make sure the .pth file is in the same folder as this script)
model.load_state_dict(torch.load("drought_gru_model.pth", map_location=torch.device('cpu')))

# 3. CRITICAL: Turn on "Evaluation Mode"
# This tells the AI: "Stop acting like a student, you are taking a test now."
model.eval() 

# ---------------------------------------------------------
# STEP 3: Prepare the New Weather Data
# ---------------------------------------------------------
# For the AI to read it, the data shape MUST be exactly:
# (1 batch, 7 days, 18 features)

# Here is an example of fake, random weather data for 1 week:
# In the real Streamlit app, your teammate will replace this with real user input.
recent_7_days_weather = np.random.rand(1, 7, 18).astype(np.float32)

# Convert the Python numbers into a PyTorch Math Tensor
new_data_tensor = torch.tensor(recent_7_days_weather)

# ---------------------------------------------------------
# STEP 4: Make the Prediction!
# ---------------------------------------------------------
# torch.no_grad() saves computer memory since we aren't training anymore
with torch.no_grad():
    # Hand the 7 days of weather to the model
    predicted_score = model(new_data_tensor)

# Extract the final number
final_drought_score = predicted_score.item()

print(f"Based on the past 7 days of weather, the predicted Drought Score is: {final_drought_score:.2f}")

Based on the past 7 days of weather, the predicted Drought Score is: 1.74
